<a href="https://colab.research.google.com/github/hayase-m/GraphRAG/blob/main/GraphRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 参考 https://blog.langchain.dev/enhancing-rag-based-applications-accuracy-by-constructing-and-leveraging-knowledge-graphs/

In [2]:
%%capture
%pip install --upgrade --quiet langchain-core langchain-community langchain-openai "langchain-neo4j>=0.10.0" langchain-text-splitters neo4j tiktoken yfiles_jupyter_graphs

In [3]:
import os
from neo4j import GraphDatabase
from yfiles_jupyter_graphs import GraphWidget
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough, ConfigurableField
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from pydantic import BaseModel, Field
from typing import List
from langchain_core.output_parsers import StrOutputParser
from langchain_neo4j import LLMGraphTransformer, Neo4jGraph, Neo4jVector
from langchain_neo4j.vectorstores.neo4j_vector import remove_lucene_chars
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import TokenTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

try:
    import google.colab
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

/tmp/ipykernel_2088/2911937224.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [4]:
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass(prompt = 'OpenAIのAPIキーを入力してください')
os.environ["NEO4J_URI"] = getpass.getpass(prompt = 'NEO4JのURIを入力してください')
os.environ["NEO4J_USERNAME"] = getpass.getpass(prompt = 'NEO4JのUSERNAMEを入力してください')
os.environ["NEO4J_PASSWORD"] = getpass.getpass(prompt = 'NEO4Jのパスワードを入力してください')

OpenAIのAPIキーを入力してください··········
NEO4JのURIを入力してください··········
NEO4JのUSERNAMEを入力してください··········
NEO4Jのパスワードを入力してください··········


In [6]:
raw_documents = TextLoader('孫悟空少年編.txt').load()
text_splitter = TokenTextSplitter(chunk_size=512, chunk_overlap=125)
documents = text_splitter.split_documents(raw_documents)

In [8]:
llm=ChatOpenAI(temperature=0, model_name="gpt-3.5-turbo-0125")
llm_transformer = LLMGraphTransformer(llm=llm)
graph_documents = llm_transformer.convert_to_graph_documents(documents)

/usr/local/lib/python3.13/dist-packages/langchain_openai/chat_models/base.py:2763: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo-0125 since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


In [9]:
graph = Neo4jGraph(database="234bdfae")

In [10]:
graph.add_graph_documents(
    graph_documents,
    baseEntityLabel=True,
    include_source=True
)

In [11]:
# directly show the graph resulting from the given Cypher query
default_cypher = "MATCH (s)-[r:!MENTIONS]->(t) RETURN s,r,t LIMIT 50"

def showGraph(cypher: str = default_cypher):
    # create a neo4j session to run queries
    driver = GraphDatabase.driver(
        uri = os.environ["NEO4J_URI"],
        auth = (os.environ["NEO4J_USERNAME"],
                os.environ["NEO4J_PASSWORD"]))
    session = driver.session()
    widget = GraphWidget(graph = session.run(cypher).graph())
    widget.node_label_mapping = 'id'
    #display(widget)
    return widget

In [12]:
showGraph()

GraphWidget(layout=Layout(height='660px', width='100%'))

In [13]:
vector_index = Neo4jVector.from_existing_graph(
    OpenAIEmbeddings(),
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
    database="234bdfae"
)

In [14]:
graph.query("CREATE FULLTEXT INDEX entity IF NOT EXISTS FOR (e:__Entity__) ON EACH [e.id]")

[]

In [15]:
# Extract entities from text
class Entities(BaseModel):
    """Identifying information about entities."""

    names: List[str] = Field(
        ...,
        description="All the person, organization, or business entities that "
        "appear in the text",
    )

In [16]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are extracting organization and person entities from the text.",
        ),
        (
            "human",
            "Use the given format to extract information from the following "
            "input: {question}",
        ),
    ]
)

entity_chain = prompt | llm.with_structured_output(Entities)

/usr/local/lib/python3.13/dist-packages/langchain_openai/chat_models/base.py:2763: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo-0125 since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


In [17]:
entity_chain.invoke({"question": "梧空と'はやせ'は戦った"}).names

['梧空', 'はやせ']

In [18]:
def generate_full_text_query(input: str) -> str:
    """
    Generate a full-text search query for a given input string.

    This function constructs a query string suitable for a full-text search.
    It processes the input string by splitting it into words and appending a
    similarity threshold (~2 changed characters) to each word, then combines
    them using the AND operator. Useful for mapping entities from user questions
    to database values, and allows for some misspelings.
    """
    full_text_query = ""
    words = [el for el in remove_lucene_chars(input).split() if el]
    for word in words[:-1]:
        full_text_query += f" {word}~2 AND"
    full_text_query += f" {words[-1]}~2"
    return full_text_query.strip()

# Fulltext index query
def structured_retriever(question: str) -> str:
    """
    Collects the neighborhood of entities mentioned
    in the question
    """
    result = ""
    entities = entity_chain.invoke({"question": question})
    for entity in entities.names:
        response = graph.query(
            """CALL db.index.fulltext.queryNodes('entity', $query, {limit:20})
            YIELD node,score
            CALL {
              WITH node
              MATCH (node)-[r:!MENTIONS]->(neighbor)
              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
              UNION ALL
              WITH node
              MATCH (node)<-[r:!MENTIONS]-(neighbor)
              RETURN neighbor.id + ' - ' + type(r) + ' -> ' +  node.id AS output
            }
            RETURN output LIMIT 1000
            """,
            {"query": generate_full_text_query(entity)},
        )
        result += "\n".join([el['output'] for el in response])
    return result

In [19]:
print(structured_retriever("孫悟空と関わりがあるエンティティを知りたい"))

孫悟飯 - POSSESSED -> 四星球
悟空 - MENTOR -> 亀仙人
悟空 - FRIEND -> クリリン
悟空 - PARTICIPANT -> 天下一武道会
悟空 - OPPONENT -> レッドリボン軍
悟空 - TRAINING -> 仙猫カリン
悟空 - REUNION -> 孫悟飯
悟空 - COMPETITOR -> 天津飯
悟空 - ENCOUNTER -> ピッコロ大魔王
悟空 - PARTICIPATED_IN -> 天下一武道会
悟空 - FOUGHT_AGAINST -> レッドリボン軍
悟空 - FOUGHT_AGAINST -> 天津飯
悟空 - TRAINING_UNDER -> 仙猫カリン
悟空 - REUNITED_WITH -> 孫悟飯
孫悟空 - ACQUAINTANCE -> ブルマ
孫悟空 - ACQUAINTANCE -> ウーロン
孫悟空 - ACQUAINTANCE -> ヤムチャ
孫悟空 - POSSESSES -> 四星球
孫悟空 - POSSESSED -> 神龍
孫悟空 - ENEMY -> ピラフ
孫悟空 - TRAINED_UNDER -> 亀仙人
孫悟空 - FRIEND -> ブルマ
孫悟空 - ADOPTED -> 孫悟飯
孫悟空 - ENCOUNTERED -> ウーロン
孫悟空 - ENCOUNTERED -> ヤムチャ
孫悟空 - ENCOUNTERED -> ピラフ
孫悟空 - TRAINING_PARTNER -> 亀仙人
悟空 - REUNION -> 孫悟飯
孫悟空 - ADOPTED -> 孫悟飯
悟空 - REUNITED_WITH -> 孫悟飯


In [20]:
def retriever(question: str):
    print(f"Search query: {question}")
    structured_data = structured_retriever(question)
    unstructured_data = [el.page_content for el in vector_index.similarity_search(question)]
    final_data = f"""Structured data:
    {structured_data}
    Unstructured data:
    {"#Document ". join(unstructured_data)}
    """
    # print(final_data)
    return final_data

In [21]:
_search_query = RunnableLambda(lambda x: x["question"])

In [22]:
template = """あなたは優秀なAIです。下記のコンテキストを利用してユーザーの質問に丁寧に答えてください。
必ず文脈からわかる情報のみを使用して回答を生成してください。
{context}

ユーザーの質問: {question}"""
prompt = ChatPromptTemplate.from_template(template)

chain = (
    RunnableParallel(
        {
            "context": _search_query | retriever,
            "question": RunnablePassthrough(),
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

In [23]:
chain.invoke({"question": "梧空と仲が悪いのは誰？"})

Search query: 梧空と仲が悪いのは誰？


'悟空と仲が悪いのはピラフです。'

## MemGraphRAGによるグラフ構築


In [24]:
%%capture
from pathlib import Path
from collections import Counter, defaultdict, deque
import json
import os
import subprocess

AIRA_DIR = Path("/content/aira-synapse")
AIRA_COMMIT = "837c2f21188d74413f1fba169cbc8c397c1d3e8b"
MEMGRAPHRAG_DIR = Path("/content/memgraphrag-demo")
MEMGRAPHRAG_DIR.mkdir(parents=True, exist_ok=True)

node_version = subprocess.check_output(["node", "--version"], text=True).strip()
if int(node_version.lstrip("v").split(".")[0]) < 20:
    raise RuntimeError(f"MemGraphRAGにはNode.js 20以上が必要です: {node_version}")

if not (AIRA_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", "--no-checkout",
         "https://github.com/nahisaho/aira-synapse.git", str(AIRA_DIR)],
        check=True,
    )
subprocess.run(
    ["git", "fetch", "--depth", "1", "origin", AIRA_COMMIT],
    cwd=AIRA_DIR,
    check=True,
)
subprocess.run(
    ["git", "checkout", "--detach", AIRA_COMMIT],
    cwd=AIRA_DIR,
    check=True,
)

subprocess.run(["npm", "ci"], cwd=AIRA_DIR, check=True)
subprocess.run(
    ["npm", "run", "build", "--workspace=packages/memgraphrag"],
    cwd=AIRA_DIR,
    check=True,
)

In [25]:
SOURCE_TEXT_PATH = Path("孫悟空少年編.txt")
if not SOURCE_TEXT_PATH.exists():
    raise FileNotFoundError("孫悟空少年編.txt をColabへアップロードしてください")
if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("先にOPENAI_API_KEYを入力してください")

MEMGRAPHRAG_CONFIG_PATH = MEMGRAPHRAG_DIR / "memgraphrag.yml"
MEMGRAPHRAG_GRAPH_PATH = MEMGRAPHRAG_DIR / "graph.json"
MEMGRAPHRAG_CONFIG_PATH.write_text(
    """version: 1
local_only: false
data_dir: /content/memgraphrag-demo/data

algorithms:
  schema:
    stabilization_threshold: 2
    canonicalization_threshold: 0.9
  conflict:
    similarity_threshold: 0.8
    scan_candidate_limit: 100
  bridging:
    similarity_threshold: 0.7
    candidate_limit: 50
  retrieval:
    top_k: 10
    top_m: 5
    threshold: 0.5
    ppr:
      teleport_probability: 0.5
      passage_damping: 0.05
      convergence_epsilon: 1.0e-6
      max_iterations: 50

chunking:
  chunk_size_tokens: 600
  chunk_overlap_tokens: 100
  context_token_limit: 8000

providers:
  llm:
    backend: openai
    model: gpt-4o-mini
    temperature: 0.1
    max_tokens: 2048
  embedding:
    backend: openai
    model: text-embedding-3-small
    dimensions: 1536
    cache_dir: /content/memgraphrag-demo/cache/embeddings
  nlp:
    backend: regex
    request_timeout_ms: 30000
    healthcheck_timeout_ms: 5000

storage:
  backend: sqlite
  sqlite_path: /content/memgraphrag-demo/memgraphrag.sqlite
  vector_index_dir: /content/memgraphrag-demo/vectors
  wal_mode: true
  auto_migrate: true

security:
  redact_stack_traces: true
  corpus_isolation: strict

limits:
  document_max_bytes: 10485760
  batch_max_documents: 100

logging:
  level: info
  audit_log_path: /content/memgraphrag-demo/audit.jsonl
  structured_log_path: /content/memgraphrag-demo/runtime.jsonl
""",
    encoding="utf-8",
)

print(f"入力ファイル: {SOURCE_TEXT_PATH}")
print(f"MemGraphRAG設定: {MEMGRAPHRAG_CONFIG_PATH}")

入力ファイル: 孫悟空少年編.txt
MemGraphRAG設定: /content/memgraphrag-demo/memgraphrag.yml


In [26]:
MEMGRAPHRAG_SCRIPT_PATH = MEMGRAPHRAG_DIR / "build_graph.mjs"
MEMGRAPHRAG_SCRIPT_PATH.write_text(
    r"""import { basename } from 'node:path';
import { readFileSync, writeFileSync } from 'node:fs';
import { createMemGraphRagRuntime, SERVICE_TOKENS } from '/content/aira-synapse/packages/memgraphrag/dist/interface/runtime/MemGraphRagRuntime.js';
import { loadMemGraphRagConfig, resolveConfigFromEnv } from '/content/aira-synapse/packages/memgraphrag/dist/infrastructure/config/index.js';

const [configPath, inputPath, outputPath] = process.argv.slice(2);
if (!configPath || !inputPath || !outputPath) {
  throw new Error('configPath, inputPath, outputPath が必要です');
}

const config = resolveConfigFromEnv(loadMemGraphRagConfig(configPath));
const runtime = createMemGraphRagRuntime(config);
await runtime.start();

try {
  const corpusManager = runtime.getService(SERVICE_TOKENS.CORPUS_MANAGER);
  const indexingService = runtime.getService(SERVICE_TOKENS.INDEXING_SERVICE);
  const graphStore = runtime.getService(SERVICE_TOKENS.GRAPH_STORE);
  const corpusName = 'songoku-memgraphrag-demo';

  const corpora = await corpusManager.list();
  const previous = corpora.find((corpus) => corpus.name === corpusName);
  if (previous) {
    await corpusManager.delete(previous.corpusId);
  }

  const corpus = await corpusManager.create(
    corpusName,
    '孫悟空少年編.txtを用いたGraphRAG比較',
  );
  const document = {
    documentId: 'songoku-shonen-txt',
    markdown: readFileSync(inputPath, 'utf8'),
    title: basename(inputPath),
    sourceUrl: inputPath,
    sourceType: 'md',
    language: 'ja',
  };

  const job = await indexingService.start({
    corpusId: corpus.corpusId,
    documents: [document],
  });
  await indexingService.resume(job.jobId);

  const jobStatus = await corpusManager.getJobStatus(job.jobId);
  if (jobStatus.status !== 'completed') {
    throw new Error(`インデックス作成に失敗しました: ${JSON.stringify(jobStatus.errors)}`);
  }

  const nodes = await graphStore.getNodes(corpus.corpusId);
  const edges = await graphStore.getEdges(corpus.corpusId);
  const graph = {
    corpusId: corpus.corpusId,
    nodes: nodes.map((node) => ({
      id: node.nodeId,
      layer: node.layer,
      label: node.label,
    })),
    edges: edges.map((edge) => ({
      source: edge.sourceNodeId,
      target: edge.targetNodeId,
      relation: edge.relation,
      weight: edge.weight,
    })),
  };
  writeFileSync(outputPath, JSON.stringify(graph, null, 2), 'utf8');

  const stats = await corpusManager.getStats(corpus.corpusId);
  console.log(JSON.stringify({
    status: jobStatus.status,
    documents: stats.documentCount,
    nodes: graph.nodes.length,
    edges: graph.edges.length,
  }, null, 2));
} finally {
  await runtime.shutdown();
}
""",
    encoding="utf-8",
)

result = subprocess.run(
    [
        "node",
        str(MEMGRAPHRAG_SCRIPT_PATH),
        str(MEMGRAPHRAG_CONFIG_PATH),
        str(SOURCE_TEXT_PATH.resolve()),
        str(MEMGRAPHRAG_GRAPH_PATH),
    ],
    cwd=AIRA_DIR,
    env=os.environ.copy(),
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("MemGraphRAGのグラフ構築に失敗しました")

  [孫悟空少年編.txt] Stage V: dict=17 thesaurus=0 ambiguous=0
  [孫悟空少年編.txt] chunks=1 schemas=18 facts=18 nodes=54 edges=71 conflicts=0
{
  "status": "completed",
  "nodes": 46,
  "edges": 71
}



In [27]:
from IPython.display import display
from ipywidgets import Layout
from yfiles_jupyter_graphs import Edge, GraphWidget, Node, NodeStyle

mem_graph = json.loads(MEMGRAPHRAG_GRAPH_PATH.read_text(encoding="utf-8"))
node_by_id = {node["id"]: node for node in mem_graph["nodes"]}
layer_counts = Counter(node["layer"] for node in mem_graph["nodes"])
relation_counts = Counter(edge["relation"] for edge in mem_graph["edges"])

print("ノード数")
for layer, count in layer_counts.most_common():
    print(f"  {layer}: {count}")
print("エッジ数")
for relation, count in relation_counts.most_common():
    print(f"  {relation}: {count}")

adjacency = defaultdict(set)
degree = Counter()
for edge in mem_graph["edges"]:
    source, target = edge["source"], edge["target"]
    adjacency[source].add(target)
    adjacency[target].add(source)
    degree[source] += 1
    degree[target] += 1

max_nodes = 250
if len(mem_graph["nodes"]) <= max_nodes:
    selected_ids = set(node_by_id)
else:
    fact_seeds = sorted(
        (node["id"] for node in mem_graph["nodes"] if node["layer"] == "fact"),
        key=lambda node_id: degree[node_id],
        reverse=True,
    )[:10]
    seed_ids = fact_seeds or sorted(node_by_id, key=lambda node_id: degree[node_id], reverse=True)[:10]
    queue = deque(seed_ids)
    selected_ids = set(seed_ids)
    while queue and len(selected_ids) < max_nodes:
        current = queue.popleft()
        for neighbor in sorted(adjacency[current], key=lambda node_id: degree[node_id], reverse=True):
            if neighbor in selected_ids:
                continue
            selected_ids.add(neighbor)
            queue.append(neighbor)
            if len(selected_ids) >= max_nodes:
                break

visible_nodes = [node for node in mem_graph["nodes"] if node["id"] in selected_ids]
visible_edges = [
    edge for edge in mem_graph["edges"]
    if edge["source"] in selected_ids and edge["target"] in selected_ids
]

nodes = []
for node in visible_nodes:
    full_label = " ".join(node["label"].split())
    display_label = full_label if len(full_label) <= 70 else full_label[:67] + "..."
    nodes.append(
        Node(
            id=node["id"],
            properties={
                "label": display_label,
                "full_label": full_label,
                "layer": node["layer"],
            },
        )
    )
edges = [
    Edge(
        start=edge["source"],
        end=edge["target"],
        properties={"label": edge["relation"], "weight": edge["weight"]},
    )
    for edge in visible_edges
]

layer_colors = {
    "ontology": "#6C5CE7",
    "fact": "#E17055",
    "passage": "#0984E3",
    "entity": "#00B894",
}
widget = GraphWidget(
    nodes=nodes,
    edges=edges,
    directed=True,
    node_label_mapping="label",
    edge_label_mapping="label",
    widget_layout=Layout(height="720px", width="100%"),
)
widget.node_styles_mapping = lambda node: NodeStyle(
    color=layer_colors.get(node["properties"].get("layer"), "#636E72")
)
print("色: ontology=紫 / fact=橙 / passage=青 / entity=緑")
print(f"表示範囲: {len(visible_nodes)}ノード / {len(visible_edges)}エッジ")
display(widget)

ノード数
  fact: 18
  entity: 17
  ontology: 10
  passage: 1
エッジ数
  entity_cooccurrence: 18
  fact_evidence: 18
  schema_instance: 18
  entity_mention: 17
色: ontology=紫 / fact=橙 / passage=青 / entity=緑
表示範囲: 46ノード / 71エッジ


GraphWidget(layout=Layout(height='720px', width='100%'))

### GraphRAGとの違い

- **GraphRAG**: 人物や対象をノード、抽出した関係をエッジとして表します。
- **MemGraphRAG**: 関係の型を表すオントロジー、文章から抽出したファクト、その根拠となるパッセージを対応付け、エンティティとパッセージ間の接続も保持します。
- `schema_instance` はオントロジーとファクト、`fact_evidence` はファクトと根拠パッセージを結びます。これにより、複数の文章断片から得た知識を共有メモリとして接続できます。